# Homework 7: Mạng LSTM (Long Short-Term Memory) để Dự đoán Tên
## Môn học: Trí Tuệ Nhân Tạo - EE3063

Nhóm Thực Hiện: [Điền tên thành viên nhóm]
MSSV: [Điền MSSV thành viên nhóm]

Nội dung:
1. Giới thiệu về LSTM và ưu điểm so với RNN đơn giản.
2. Chuẩn bị dữ liệu tên (tương tự HW6).
3. Xây dựng mô hình LSTM đơn giản từ đầu.
4. Huấn luyện mô hình LSTM.
5. Dự đoán ký tự và sinh tên bằng LSTM.

In [60]:
# -*- coding: utf-8 -*-
import numpy as np

## 1. Giới thiệu về LSTM

**Long Short-Term Memory (LSTM)** là một kiến trúc mạng nơ-ron hồi quy (RNN) đặc biệt, được thiết kế để giải quyết các vấn đề của RNN truyền thống, đặc biệt là vấn đề **vanishing gradient** (gradient biến mất) và khả năng học các **phụ thuộc xa (long-term dependencies)** trong dữ liệu chuỗi.

### Cấu trúc của một ô LSTM (LSTM Cell)

Điểm cốt lõi của LSTM là **ô nhớ (cell)**, có khả năng duy trì thông tin qua nhiều bước thời gian. Trạng thái của ô nhớ, gọi là **cell state ($C_t$)**, được điều khiển bởi ba loại cổng (gates):

1.  **Cổng Quên (Forget Gate - $f_t$):** Quyết định thông tin nào từ cell state trước đó ($C_{t-1}$) sẽ bị loại bỏ. Nó xem xét $h_{t-1}$ (output của bước trước) và $x_t$ (input hiện tại), và đưa ra một số từ 0 đến 1 cho mỗi số trong cell state $C_{t-1}$. 1 nghĩa là "giữ hoàn toàn", 0 nghĩa là "quên hoàn toàn".
    $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$

2.  **Cổng Vào (Input Gate - $i_t$):** Quyết định thông tin mới nào sẽ được lưu trữ trong cell state. Bao gồm hai phần:
    * Một lớp sigmoid ($i_t$) quyết định giá trị nào sẽ được cập nhật.
        $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$
    * Một lớp tanh ($\tilde{C}_t$) tạo ra một vector các giá trị ứng viên mới có thể được thêm vào cell state.
        $\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$

3.  **Cập nhật Cell State:** Cell state cũ $C_{t-1}$ được cập nhật thành $C_t$:
    $C_t = f_t * C_{t-1} + i_t * \tilde{C}_t$
    (Phần cũ được nhân với $f_t$, phần ứng viên mới được nhân với $i_t$).

4.  **Cổng Đầu Ra (Output Gate - $o_t$):** Quyết định phần nào của cell state sẽ được đưa ra làm output (hidden state $h_t$).
    * Một lớp sigmoid ($o_t$) quyết định phần nào của cell state sẽ được output.
        $o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$
    * Cell state được đưa qua hàm $\tanh$ (để giá trị nằm trong khoảng [-1, 1]) và sau đó nhân với output của cổng sigmoid.
        $h_t = o_t * \tanh(C_t)$

Trong đó, $\sigma$ là hàm sigmoid, và $[h_{t-1}, x_t]$ là phép nối (concatenation) của hidden state trước đó và input hiện tại. $W_f, W_i, W_C, W_o$ và $b_f, b_i, b_C, b_o$ là các ma trận trọng số và vector bias tương ứng được học trong quá trình huấn luyện.

### Ưu điểm của LSTM
-   Khả năng ghi nhớ thông tin trong thời gian dài tốt hơn RNN đơn giản.
-   Giảm thiểu vấn đề vanishing/exploding gradient.

(Tham khảo slide: "LSTM.pdf")

## 2. Chuẩn bị Dữ liệu One-Hot Encoding (Tương tự HW6)

1.  **Biểu diễn Input bằng One-Hot Encoding:**
    * Mỗi ký tự đầu vào tại mỗi bước thời gian ($x_t$) sẽ được chuyển đổi thành một **vector cột one-hot** ngay từ giai đoạn chuẩn bị dữ liệu.
    * Chuỗi đầu vào cho hàm `train_sequence` sẽ là một danh sách các vector one-hot này.
    * Hàm `forward_step` của LSTM sẽ nhận trực tiếp vector one-hot làm đầu vào cho các phép tính ma trận.
    * CCó ví dụ in ra để minh họa cách một chuỗi ký tự được chuyển thành chuỗi các vector one-hot.

2.  **Cơ chế Sinh Tên chỉ từ Ký tự Đầu:**
    * Hàm `generate_name_lstm` sẽ chỉ sử dụng ký tự đầu tiên (dưới dạng one-hot) làm input thực sự.
    * Các bước sinh ký tự sau đó sẽ sử dụng một input "giả" (vector zero dạng cột). Mô hình phải dựa vào thông tin lưu trữ trong trạng thái ẩn ($h_t$) và trạng thái ô nhớ ($C_t$) để tiếp tục sinh ra phần còn lại của tên.


In [61]:
data_names = ["Bình", "Long", "Dũng"]

char_set = set()  
for name in data_names:
    for char_val in name: 
        char_set.add(char_val)

sorted_chars = sorted(list(char_set)) 
char_to_int_map = {ch: i for i, ch in enumerate(sorted_chars)} 
int_to_char_map = {i: ch for i, ch in enumerate(sorted_chars)} 
vocab_size = len(sorted_chars) 

print(f"Bộ từ vựng ({vocab_size} ký tự): {sorted_chars}")
print(f"Ánh xạ ký tự sang số: {char_to_int_map}")

def char_to_one_hot_vector(char_idx, current_vocab_size): 
    """Chuyển chỉ số ký tự thành vector one-hot dạng cột."""
    vec = np.zeros((current_vocab_size, 1))
    vec[char_idx, 0] = 1
    return vec

training_data_lstm = []
for name_str_item in data_names: 
    if len(name_str_item) > 1:
        input_one_hot_seq = []
        target_indices_seq = []
        for i in range(len(name_str_item) - 1):
            input_char_item = name_str_item[i] 
            target_char_item = name_str_item[i+1] 
            input_one_hot_seq.append(char_to_one_hot_vector(char_to_int_map[input_char_item], vocab_size))
            target_indices_seq.append(char_to_int_map[target_char_item])
        if input_one_hot_seq:
            training_data_lstm.append((input_one_hot_seq, target_indices_seq))

# --- In ví dụ One-Hot Encoding ---
print("\n--- Ví dụ One-Hot Encoding cho tên đầu tiên trong dữ liệu huấn luyện (LSTM) ---")
if training_data_lstm:
    sample_name_to_print = data_names[0] 
    sample_input_one_hot_to_print, sample_target_indices_to_print = training_data_lstm[0] 
    print(f"Tên mẫu: '{sample_name_to_print}'")
    print("Chuỗi input (one-hot) và target (index) tương ứng:")
    for t_step in range(len(sample_input_one_hot_to_print)): 
        input_char_print = sample_name_to_print[t_step] 
        target_char_print = sample_name_to_print[t_step+1] 
        print(f"  Bước {t_step}:")
        print(f"    Input ký tự: '{input_char_print}' -> Vector One-Hot (dạng cột, {vocab_size}x1):\n{sample_input_one_hot_to_print[t_step].T}")
        print(f"    Target ký tự: '{target_char_print}' -> Index: {sample_target_indices_to_print[t_step]}")
else:
    print("Không có dữ liệu huấn luyện để hiển thị ví dụ one-hot.")

Bộ từ vựng (9 ký tự): ['B', 'D', 'L', 'g', 'h', 'n', 'o', 'ì', 'ũ']
Ánh xạ ký tự sang số: {'B': 0, 'D': 1, 'L': 2, 'g': 3, 'h': 4, 'n': 5, 'o': 6, 'ì': 7, 'ũ': 8}

--- Ví dụ One-Hot Encoding cho tên đầu tiên trong dữ liệu huấn luyện (LSTM) ---
Tên mẫu: 'Bình'
Chuỗi input (one-hot) và target (index) tương ứng:
  Bước 0:
    Input ký tự: 'B' -> Vector One-Hot (dạng cột, 9x1):
[[1. 0. 0. 0. 0. 0. 0. 0. 0.]]
    Target ký tự: 'ì' -> Index: 7
  Bước 1:
    Input ký tự: 'ì' -> Vector One-Hot (dạng cột, 9x1):
[[0. 0. 0. 0. 0. 0. 0. 1. 0.]]
    Target ký tự: 'n' -> Index: 5
  Bước 2:
    Input ký tự: 'n' -> Vector One-Hot (dạng cột, 9x1):
[[0. 0. 0. 0. 0. 1. 0. 0. 0.]]
    Target ký tự: 'h' -> Index: 4


## 3. Xây dựng Mô hình LSTM

In [62]:
def sigmoid_fn(x): return 1 / (1 + np.exp(-x)) 
def tanh_fn(x): return np.tanh(x) 
def softmax_fn(x): 
    e_x = np.exp(x - np.max(x, axis=0, keepdims=True))
    return e_x / np.sum(e_x, axis=0, keepdims=True)

class SimpleLSTMModel: #  lớp
    def __init__(self, current_vocab_size, current_hidden_size, current_learning_rate=0.01): 
        self.vocab_size = current_vocab_size
        self.hidden_size = current_hidden_size
        self.lr = current_learning_rate
        concat_size = self.hidden_size + self.vocab_size # Sửa lại cách tham chiếu
        
        # Khởi tạo trọng số
        self.W_f = np.random.randn(self.hidden_size, concat_size) * 0.01; self.b_f = np.zeros((self.hidden_size, 1))
        self.W_i = np.random.randn(self.hidden_size, concat_size) * 0.01; self.b_i = np.zeros((self.hidden_size, 1))
        self.W_c_candidate = np.random.randn(self.hidden_size, concat_size) * 0.01; self.b_c_candidate = np.zeros((self.hidden_size, 1)) #  W_c, b_c
        self.W_o = np.random.randn(self.hidden_size, concat_size) * 0.01; self.b_o = np.zeros((self.hidden_size, 1))
        self.W_y_output = np.random.randn(self.vocab_size, self.hidden_size) * 0.01; self.b_y_output = np.zeros((self.vocab_size, 1)) #  W_y, b_y

    def forward_step(self, x_input_one_hot, h_prev, C_prev): 
        concat_combined_input = np.vstack((h_prev, x_input_one_hot)) 
        
        f_gate = sigmoid_fn(np.dot(self.W_f, concat_combined_input) + self.b_f) 
        i_gate = sigmoid_fn(np.dot(self.W_i, concat_combined_input) + self.b_i) 
        C_tilde_candidate = tanh_fn(np.dot(self.W_c_candidate, concat_combined_input) + self.b_c_candidate) 
        
        C_current = f_gate * C_prev + i_gate * C_tilde_candidate 
        
        o_gate = sigmoid_fn(np.dot(self.W_o, concat_combined_input) + self.b_o) 
        h_current = o_gate * tanh_fn(C_current) 
        
        output_logits = np.dot(self.W_y_output, h_current) + self.b_y_output 
        y_predicted_proba = softmax_fn(output_logits) 
        
        # Cache cho BPTT
        cache_data = { 
            'concat_input': concat_combined_input, 'f_t': f_gate, 'i_t': i_gate, 'C_tilde_t': C_tilde_candidate,
            'C_prev': C_prev, 'C_next': C_current, 'o_t': o_gate, 
            'h_prev': h_prev, 'h_next': h_current,
            'o_logits': output_logits, 'y_pred_proba': y_predicted_proba
        }
        return y_predicted_proba, h_current, C_current, cache_data

    def train_single_sequence(self, input_one_hot_seq, target_idx_seq): 
        loss_val = 0 
        h_prev_state = np.zeros((self.hidden_size, 1)) 
        C_prev_state = np.zeros((self.hidden_size, 1)) 
        sequence_caches = [] 

        for t_idx in range(len(input_one_hot_seq)): 
            x_one_hot_current = input_one_hot_seq[t_idx] 
            y_target_current_idx = target_idx_seq[t_idx] 
            
            y_pred_p, h_next_s, C_next_s, cache_item = self.forward_step(x_one_hot_current, h_prev_state, C_prev_state) 
            sequence_caches.append(cache_item)
            
            loss_t_val = -np.log(y_pred_p[y_target_current_idx, 0] + 1e-9) 
            loss_val += loss_t_val
            
            h_prev_state, C_prev_state = h_next_s, C_next_s
            
        avg_loss_val = loss_val / len(input_one_hot_seq) 

        # BPTT
        dWf, dWi, dWc_cand, dWo, dWy_out = (np.zeros_like(p) for p in [self.W_f, self.W_i, self.W_c_candidate, self.W_o, self.W_y_output]) 
        dbf, dbi, dbc_cand, dbo, dby_out = (np.zeros_like(p) for p in [self.b_f, self.b_i, self.b_c_candidate, self.b_o, self.b_y_output]) 
        dh_next_grad = np.zeros_like(h_prev_state) 
        dC_next_grad = np.zeros_like(C_prev_state) 

        for t_idx_rev in reversed(range(len(input_one_hot_seq))): 
            cache_item_rev = sequence_caches[t_idx_rev] 
            y_target_current_idx_rev = target_idx_seq[t_idx_rev] 
            
            dy_logits_grad = np.copy(cache_item_rev['y_pred_proba']) 
            dy_logits_grad[y_target_current_idx_rev] -= 1
            
            dWy_out += np.dot(dy_logits_grad, cache_item_rev['h_next'].T)
            dby_out += dy_logits_grad
            
            dh_t_grad = np.dot(self.W_y_output.T, dy_logits_grad) + dh_next_grad 
            
            do_gate_grad = dh_t_grad * tanh_fn(cache_item_rev['C_next']) 
            do_gate_raw_grad = do_gate_grad * cache_item_rev['o_t'] * (1 - cache_item_rev['o_t']) 
            dWo += np.dot(do_gate_raw_grad, cache_item_rev['concat_input'].T)
            dbo += do_gate_raw_grad
            
            dC_t_grad = dh_t_grad * cache_item_rev['o_t'] * (1 - tanh_fn(cache_item_rev['C_next'])**2) + dC_next_grad 
            dC_prev_grad_for_next_iter = dC_t_grad * cache_item_rev['f_t'] 

            df_gate_grad = dC_t_grad * cache_item_rev['C_prev'] 
            df_gate_raw_grad = df_gate_grad * cache_item_rev['f_t'] * (1 - cache_item_rev['f_t']) 
            dWf += np.dot(df_gate_raw_grad, cache_item_rev['concat_input'].T)
            dbf += df_gate_raw_grad
            
            di_gate_grad = dC_t_grad * cache_item_rev['C_tilde_t'] 
            di_gate_raw_grad = di_gate_grad * cache_item_rev['i_t'] * (1 - cache_item_rev['i_t']) 
            dWi += np.dot(di_gate_raw_grad, cache_item_rev['concat_input'].T)
            dbi += di_gate_raw_grad
            
            dC_tilde_cand_grad = dC_t_grad * cache_item_rev['i_t'] 
            dC_tilde_cand_raw_grad = dC_tilde_cand_grad * (1 - cache_item_rev['C_tilde_t']**2) 
            dWc_cand += np.dot(dC_tilde_cand_raw_grad, cache_item_rev['concat_input'].T)
            dbc_cand += dC_tilde_cand_raw_grad
            
            d_concat_combined_input_grad = (np.dot(self.W_f.T, df_gate_raw_grad) + 
                                          np.dot(self.W_i.T, di_gate_raw_grad) +
                                          np.dot(self.W_c_candidate.T, dC_tilde_cand_raw_grad) +
                                          np.dot(self.W_o.T, do_gate_raw_grad))
            
            dh_next_grad = d_concat_combined_input_grad[:self.hidden_size, :]
            dC_next_grad = dC_prev_grad_for_next_iter

        model_gradients = [dWf, dWi, dWc_cand, dWo, dWy_out, dbf, dbi, dbc_cand, dbo, dby_out] 
        for d_param in model_gradients: np.clip(d_param, -5, 5, out=d_param) 
        
        calculated_gradients = { 
            'dW_f': dWf, 'dW_i': dWi, 'dW_c': dWc_cand, 'dW_o': dWo, 'dW_y': dWy_out,
            'db_f': dbf, 'db_i': dbi, 'db_c': dbc_cand, 'db_o': dbo, 'db_y': dby_out
        }
        return avg_loss_val, calculated_gradients

    def apply_gradients(self, calculated_gradients): 
        self.W_f -= self.lr * calculated_gradients['dW_f']; self.b_f -= self.lr * calculated_gradients['db_f']
        self.W_i -= self.lr * calculated_gradients['dW_i']; self.b_i -= self.lr * calculated_gradients['db_i']
        self.W_c_candidate -= self.lr * calculated_gradients['dW_c']; self.b_c_candidate -= self.lr * calculated_gradients['db_c']
        self.W_o -= self.lr * calculated_gradients['dW_o']; self.b_o -= self.lr * calculated_gradients['db_o']
        self.W_y_output -= self.lr * calculated_gradients['dW_y']; self.b_y_output -= self.lr * calculated_gradients['db_y']


## 4. Huấn luyện Mô hình LSTM

In [63]:
lstm_hidden_size = 30 
lstm_learning_rate = 0.01
lstm_num_epochs = 5000

lstm_instance = SimpleLSTMModel(vocab_size, lstm_hidden_size, lstm_learning_rate) 

print(f"\nBắt đầu huấn luyện LSTM (Cleaned Names) với hidden_size={lstm_hidden_size}, lr={lstm_learning_rate}...")
for epoch_num in range(lstm_num_epochs): 
    epoch_total_loss = 0 
    for input_seq_oh, target_seq_idx in training_data_lstm: 
        current_loss, current_grads = lstm_instance.train_single_sequence(input_seq_oh, target_seq_idx) 
        lstm_instance.apply_gradients(current_grads)
        epoch_total_loss += current_loss
    avg_epoch_loss = epoch_total_loss / len(training_data_lstm) if training_data_lstm else 0 
    if (epoch_num + 1) % 400 == 0:
        print(f"Epoch {epoch_num+1}/{lstm_num_epochs}, Loss trung bình LSTM: {avg_epoch_loss:.4f}")
print("Hoàn tất huấn luyện LSTM (Cleaned Names).")


Bắt đầu huấn luyện LSTM (Cleaned Names) với hidden_size=30, lr=0.01...
Epoch 400/5000, Loss trung bình LSTM: 1.7222
Epoch 800/5000, Loss trung bình LSTM: 1.4672
Epoch 1200/5000, Loss trung bình LSTM: 0.9010
Epoch 1600/5000, Loss trung bình LSTM: 0.2497
Epoch 2000/5000, Loss trung bình LSTM: 0.0595
Epoch 2400/5000, Loss trung bình LSTM: 0.0280
Epoch 2800/5000, Loss trung bình LSTM: 0.0174
Epoch 3200/5000, Loss trung bình LSTM: 0.0123
Epoch 3600/5000, Loss trung bình LSTM: 0.0094
Epoch 4000/5000, Loss trung bình LSTM: 0.0076
Epoch 4400/5000, Loss trung bình LSTM: 0.0063
Epoch 4800/5000, Loss trung bình LSTM: 0.0054
Hoàn tất huấn luyện LSTM (Cleaned Names).


## 5. Dự đoán Ký tự và Sinh Tên bằng LSTM

In [66]:
def generate_name_with_lstm(lstm_model_instance, initial_char, name_max_length): 
    if initial_char not in char_to_int_map: 
        print(f"Ký tự '{initial_char}' không có trong bộ từ vựng.")
        return initial_char
        
    output_name = initial_char 
    current_h_state = np.zeros((lstm_model_instance.hidden_size, 1)) 
    current_C_state = np.zeros((lstm_model_instance.hidden_size, 1)) 
    
    # Bước 1: Xử lý ký tự đầu tiên
    initial_char_idx = char_to_int_map[initial_char] 
    current_x_one_hot = char_to_one_hot_vector(initial_char_idx, lstm_model_instance.vocab_size) 
    
    pred_proba, next_h_state, next_C_state, _ = lstm_model_instance.forward_step(current_x_one_hot, current_h_state, current_C_state) 
    current_h_state, current_C_state = next_h_state, next_C_state
    
    predicted_char_idx = np.argmax(pred_proba.flatten()) 
    output_name += int_to_char_map[predicted_char_idx] 
    
    # Các bước tiếp theo: dùng dummy input
    dummy_one_hot_input = np.zeros((lstm_model_instance.vocab_size, 1)) 
    
    for _ in range(name_max_length - len(initial_char) - 1):
        if len(output_name) >= name_max_length: break
        pred_proba, next_h_state, next_C_state, _ = lstm_model_instance.forward_step(dummy_one_hot_input, current_h_state, current_C_state)
        current_h_state, current_C_state = next_h_state, next_C_state
        predicted_char_idx = np.argmax(pred_proba.flatten())
        output_name += int_to_char_map[predicted_char_idx]
    return output_name

print("\n--- Dự đoán Tên Sinh Viên LTSM (LSTM) ---")
target_names_list = ["Bình", "Long", "Dũng"] 
initial_chars_list = ["B", "L", "D"] 

for idx in range(len(initial_chars_list)): 
    start_c = initial_chars_list[idx] 
    target_n = target_names_list[idx] 
    max_len = len(target_n) 
    generated_n_lstm = generate_name_with_lstm(lstm_instance, start_c, max_len) 
    print(f"Bắt đầu bằng '{start_c}', Hoàn thành (LSTM): '{generated_n_lstm}' (Mong muốn: '{target_n}')")



--- Dự đoán Tên Sinh Viên LTSM (LSTM) ---
Bắt đầu bằng 'B', Hoàn thành (LSTM): 'Bìnn' (Mong muốn: 'Bình')
Bắt đầu bằng 'L', Hoàn thành (LSTM): 'Lonn' (Mong muốn: 'Long')
Bắt đầu bằng 'D', Hoàn thành (LSTM): 'Dũnn' (Mong muốn: 'Dũng')
